# Phishing-Email SLM Fine-Tuning Starter (QLoRA) — Colab-Safe Version
**Project:** Small but Sufficient? — SLM vs. LLM phishing-email detection

This notebook covers **Weeks 5–6** of your project plan: naive LoRA fine-tuning and CoT-distilled LoRA fine-tuning of a small model on phishing-email classification, with checkpoints saved to **Google Drive** so a Colab disconnect never loses your trained adapter.

**How to use, step by step:**
1. Open this file in Google Colab (Colab → File → Upload notebook, or open from GitHub if you pushed the repo).
2. Runtime → Change runtime type → **T4 GPU** (free tier is enough).
3. Run Cell 1 (installs) then Cell 2 (Drive mount) — you'll get a Google auth popup, approve it.
4. Edit **Cell 4 (Config)** — this is the only cell you should need to change between runs.
5. Run the rest top to bottom. It works today on 3 synthetic placeholder rows so you can confirm the whole pipeline before your real Week 2 dataset exists.
6. Run it **twice per model**: once with `USE_COT = False` (Week 5), once with `USE_COT = True` (Week 6) — everything else stays identical so the comparison is fair.


In [ ]:
# CELL 1 — Install unpinned packages for smoke test environment audit
# Note: "datasets>=3.0.0" is quoted to prevent bash shell redirection bug
!pip install -q transformers trl peft bitsandbytes accelerate "datasets>=3.0.0" pandas scikit-learn
!pip uninstall -y wandb -q

import sys, subprocess, inspect
import torch, transformers, trl, peft, bitsandbytes, accelerate, datasets
from trl import SFTConfig

print("\n=== ENVIRONMENT & PACKAGE AUDIT ===")
print(f"Python:       {sys.version.split()[0]}")
print(f"CUDA:         {torch.version.cuda} (Available: {torch.cuda.is_available()})")

freeze_out = subprocess.check_output([sys.executable, "-m", "pip", "freeze"]).decode("utf-8")
tracked_pkgs = ["torch", "transformers", "trl", "peft", "bitsandbytes", "accelerate", "datasets"]
for line in freeze_out.splitlines():
    for pkg in tracked_pkgs:
        if line.lower().startswith(f"{pkg}=="):
            print(f"pip freeze:   {line}")

print(f"\ntrl.__version__: {trl.__version__}")
has_completion_loss = 'completion_only_loss' in inspect.signature(SFTConfig).parameters
print(f"'completion_only_loss' in inspect.signature(SFTConfig).parameters: {has_completion_loss}")
print("===================================\n")


In [ ]:
# CELL 2 — Mount Google Drive (Colab-safe checkpoint storage)
import os
IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = "/content/drive/MyDrive/phishing-slm-research"
    os.makedirs(f"{DRIVE_ROOT}/data/processed", exist_ok=True)
    os.makedirs(f"{DRIVE_ROOT}/finetuning/checkpoints", exist_ok=True)
    print("Mounted Google Drive. Project root:", DRIVE_ROOT)
else:
    DRIVE_ROOT = "."
    print("Local environment root:", DRIVE_ROOT)


In [ ]:
# CELL 3 — Imports and GPU check
import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device: ", torch.cuda.get_device_name(0))
    print("GPU Memory: ", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU.")


In [ ]:
# CELL 4 — Config (Arm A Naive QLoRA)
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
USE_COT = False               # Arm A: False = naive direct classification
SMOKE_TEST = True             # True for Step 2 smoke test (200 rows, 40 steps); False for full run
RESUME_FROM_CHECKPOINT = False # Set True ONLY to resume an interrupted full run
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

# Hardware precision decision for T4 GPU: fp16 is native and optimal on Turing
USE_FP16 = True
USE_BF16 = False

# SMOKE_TEST writes to a separate dir (...-naive-SMOKE) so full run cannot accidentally resume from smoke files
dir_suffix = "-naive-SMOKE" if SMOKE_TEST else ("-cot" if USE_COT else "-naive")
OUTPUT_DIR = f"{DRIVE_ROOT}/finetuning/checkpoints/{MODEL_NAME.split('/')[-1]}{dir_suffix}"

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
NUM_EPOCHS = 3
MAX_SUBJECT_TOKENS = 100       # Pre-format subject token budget
MAX_BODY_TOKENS = 800          # Pre-format body token budget
MAX_SEQ_LENGTH = 1024          # Explicit sequence length ceiling
SAVE_STEPS = 200               # Saves checkpoint every ~20-30 mins to protect against disconnects

print("Model:                  ", MODEL_NAME)
print("Mode:                   ", 'CoT-distilled' if USE_COT else 'Naive label-only (Arm A)')
print("Smoke test:             ", SMOKE_TEST)
print("Resume from checkpoint: ", RESUME_FROM_CHECKPOINT)
print("Output dir:             ", OUTPUT_DIR)


In [ ]:
# CELL 5 — Load training and validation data (STRICT READ-ONLY, NO WRITE/OVERWRITE)
import os, hashlib

TRAIN_PATH = f"{DRIVE_ROOT}/data/processed/phishing_train.csv"
VAL_PATH = f"{DRIVE_ROOT}/data/processed/phishing_val.csv"

def get_sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest()

if not os.path.exists(TRAIN_PATH):
    raise FileNotFoundError(f"CRITICAL: phishing_train.csv not found at {TRAIN_PATH}! Please copy it to Drive.")
if not os.path.exists(VAL_PATH):
    raise FileNotFoundError(f"CRITICAL: phishing_val.csv not found at {VAL_PATH}! Please copy it to Drive.")

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

# Assert exact row counts (test set is strictly locked and never read)
assert len(train_df) == 20681, f"Train count mismatch: expected 20,681, got {len(train_df)}"
assert len(val_df) == 2954, f"Val count mismatch: expected 2,954, got {len(val_df)}"

train_sha = get_sha256(TRAIN_PATH)
val_sha = get_sha256(VAL_PATH)

print(f"phishing_train.csv: {len(train_df)} rows | SHA256: {train_sha}")
print(f"phishing_val.csv:   {len(val_df)} rows | SHA256: {val_sha}")

df = train_df
if SMOKE_TEST:
    df = df.head(200).copy()
    print(f"SMOKE TEST ACTIVE: Subsampled to first {len(df)} rows.")


In [ ]:
# CELL 6 — Inlined Prompt Template & Pre-format Truncation (No leading space)
COMPLETION_PHISHING = "Phishing"
COMPLETION_LEGITIMATE = "Legitimate"
MAX_BODY_TOKENS = 800
MAX_SUBJECT_TOKENS = 100
MAX_SEQ_LENGTH = 1024

def truncate_text_tokens(text, max_tokens, tokenizer=None):
    if not text or not isinstance(text, str):
        return "", False
    if len(text) > max_tokens * 8:
        candidate = text[:max_tokens * 8]
        toks = tokenizer.encode(candidate, add_special_tokens=False)
        if len(toks) >= max_tokens:
            return tokenizer.decode(toks[:max_tokens], skip_special_tokens=True), True
            
    toks = tokenizer.encode(text, add_special_tokens=False)
    if len(toks) > max_tokens:
        return tokenizer.decode(toks[:max_tokens], skip_special_tokens=True), True
    return text, False

def build_prompt_pair(subject, body, label_int, tokenizer=None):
    s_trunc, s_was_trunc = truncate_text_tokens(subject, MAX_SUBJECT_TOKENS, tokenizer)
    b_trunc, b_was_trunc = truncate_text_tokens(body, MAX_BODY_TOKENS, tokenizer)
    prompt_str = (
        f"### Instruction:\n"
        f"Classify the following email as 'Phishing' or 'Legitimate'.\n\n"
        f"### Email:\n"
        f"Subject: {s_trunc}\n\n"
        f"Body:\n{b_trunc}\n\n"
        f"### Response:\n"
    )
    label_str = COMPLETION_PHISHING if label_int == 1 else COMPLETION_LEGITIMATE
    return {
        "prompt": prompt_str,
        "completion": label_str,
        "s_trunc": s_was_trunc,
        "b_trunc": b_was_trunc
    }

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Verify tokenization of completions (Point 4)
ids_p = tokenizer.encode(COMPLETION_PHISHING, add_special_tokens=False)
ids_l = tokenizer.encode(COMPLETION_LEGITIMATE, add_special_tokens=False)
print(f"'{COMPLETION_PHISHING}': Token IDs={ids_p} | Decoded={[tokenizer.decode([t]) for t in ids_p]}")
print(f"'{COMPLETION_LEGITIMATE}': Token IDs={ids_l} | Decoded={[tokenizer.decode([t]) for t in ids_l]}")
print(f"First tokens distinct? {ids_p[0] != ids_l[0]} ({ids_p[0]} vs {ids_l[0]})")
assert ids_p[0] != ids_l[0], "First tokens must be distinct!"

# Build prompt-completion dataset & enforce MAX_SEQ_LENGTH ceiling
body_truncated_count = 0
subject_truncated_count = 0
records = []
total_tokens_list = []
eos_token = tokenizer.eos_token

for _, row in df.iterrows():
    pair_data = build_prompt_pair(row['subject'], row['body'], int(row['label']), tokenizer=tokenizer)
    if pair_data['b_trunc']:
        body_truncated_count += 1
    if pair_data['s_trunc']:
        subject_truncated_count += 1
        
    full_seq = pair_data["prompt"] + pair_data["completion"] + eos_token
    tok_len = len(tokenizer.encode(full_seq, add_special_tokens=False))
    total_tokens_list.append(tok_len)
    
    records.append({"prompt": pair_data["prompt"], "completion": pair_data["completion"]})

import numpy as np
arr = np.array(total_tokens_list)
max_observed = int(np.max(arr))
rows_exceeding = int((arr > MAX_SEQ_LENGTH).sum())

print(f"\n--- Sequence Length Audit (Target max_length: {MAX_SEQ_LENGTH}) ---")
print(f"Dataset Rows:            {len(df)}")
print(f"Min Tokens:              {int(np.min(arr))}")
print(f"p50 (Median) Tokens:     {float(np.median(arr)):.1f}")
print(f"p90 Tokens:              {float(np.percentile(arr, 90)):.1f}")
print(f"p99 Tokens:              {float(np.percentile(arr, 99)):.1f}")
print(f"Max Tokens:              {max_observed}")
print(f"Rows > {MAX_SEQ_LENGTH}:            {rows_exceeding} (Must strictly be 0)")
print(f"Bodies Truncated:        {body_truncated_count} ({body_truncated_count / len(df) * 100:.2f}%)")
print(f"Subjects Truncated:      {subject_truncated_count} ({subject_truncated_count / len(df) * 100:.2f}%)")

assert rows_exceeding == 0, f"CRITICAL: {rows_exceeding} sequences exceed max_length {MAX_SEQ_LENGTH}! Max sequence was {max_observed}."

train_dataset = Dataset.from_pandas(pd.DataFrame(records))
print("\nSample prompt:", repr(train_dataset[0]['prompt'][:80]), "...")
print("Sample completion:", repr(train_dataset[0]['completion']))


In [ ]:
# CELL 7 — Load model in 4-bit (QLoRA) with fp16 compute for T4
from peft import prepare_model_for_kbit_training

target_dtype = torch.float16 if USE_FP16 else torch.bfloat16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=target_dtype,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=target_dtype,
    torch_dtype=target_dtype
)
model.config.use_cache = False

# Prepare model for k-bit training (casts layernorms to float32)
model = prepare_model_for_kbit_training(model)

# Explicitly ensure all non-quantized modules (embed_tokens, lm_head, layernorms) are float32
# so NO bfloat16 tensors exist anywhere in the model
for name, module in model.named_modules():
    if hasattr(module, 'weight') and module.weight is not None:
        if not hasattr(module, 'quant_state'):
            if module.weight.dtype == torch.bfloat16:
                module.to(torch.float32)


In [ ]:
# CELL 8 — Attach LoRA adapter (Attention + MLP projections, excluding lm_head)
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)

# Ensure all trainable LoRA parameters are float32 to prevent PyTorch GradScaler unscale crash on BFloat16
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

trainable_dtypes = set(p.dtype for p in model.parameters() if p.requires_grad)
print(f'Trainable parameter dtypes: {trainable_dtypes}')
assert torch.bfloat16 not in trainable_dtypes, f'Found BFloat16 in trainable parameters: {trainable_dtypes}'

model.print_trainable_parameters()


In [ ]:
# CELL 9 — Resumable Training with Adaptive TRL Compatibility
import time, json, inspect
from trl import SFTTrainer, SFTConfig

# 1. Self-Verification of Loss Masking on 3 examples
print("\n--- [VERIFICATION] Prompt-Completion Loss Masking Check ---")
eos_token = tokenizer.eos_token
for i in range(min(3, len(train_dataset))):
    s = train_dataset[i]
    p_text = s["prompt"]
    c_text = s["completion"] + eos_token
    p_ids = tokenizer.encode(p_text, add_special_tokens=False)
    c_ids = tokenizer.encode(c_text, add_special_tokens=False)
    full_ids = p_ids + c_ids
    labels = [-100] * len(p_ids) + c_ids
    unmasked = [idx for idx, x in enumerate(labels) if x != -100]
    unmasked_tokens = [tokenizer.decode([full_ids[idx]]) for idx in unmasked]
    print(f"Example {i+1}: {len(full_ids)} tokens | {len(p_ids)} prompt tokens masked (-100) | Trained completion: {unmasked_tokens}")
    assert len(unmasked) == len(c_ids), "Masking assertion failed!"
print("--- [VERIFIED] Prompt tokens masked (-100); only completion tokens are trained! ---\n")

# 2. Check for Existing Checkpoints to Support Resumption ONLY when explicitly flagged
last_checkpoint = None
if RESUME_FROM_CHECKPOINT and not SMOKE_TEST:
    last_checkpoint = get_last_checkpoint(OUTPUT_DIR)
    if last_checkpoint is not None:
        print(f"[RESUME]: Found previous checkpoint at {last_checkpoint}. Resuming training!")
    else:
        print(f"[RESUME]: RESUME_FROM_CHECKPOINT is True, but no checkpoint found in {OUTPUT_DIR}. Starting from scratch.")
else:
    if not SMOKE_TEST:
        print("[TRAIN]: Clean run starting from scratch (RESUME_FROM_CHECKPOINT = False).")

max_steps = 40 if SMOKE_TEST else -1
epochs = 1 if SMOKE_TEST else NUM_EPOCHS
save_strategy = "no" if SMOKE_TEST else "steps"

# Check TRL signature compatibility dynamically
has_completion_loss = 'completion_only_loss' in inspect.signature(SFTConfig).parameters
has_max_seq_length = 'max_seq_length' in inspect.signature(SFTConfig).parameters

sft_kwargs = {
    "output_dir": OUTPUT_DIR,
    "num_train_epochs": epochs,
    "max_steps": max_steps,
    "per_device_train_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "logging_steps": 5 if SMOKE_TEST else 20,
    "save_strategy": save_strategy,
    "save_steps": SAVE_STEPS if not SMOKE_TEST else 500,
    "save_total_limit": 2,
    "fp16": False,  # QLoRA uses bnb_4bit_compute_dtype=float16 internally; disabling global fp16 prevents GradScaler crash
    "bf16": False,
    "report_to": "none",
    "seed": RANDOM_SEED
}
if has_completion_loss:
    sft_kwargs["completion_only_loss"] = True
    sft_kwargs["max_length"] = MAX_SEQ_LENGTH
elif has_max_seq_length:
    sft_kwargs["max_seq_length"] = MAX_SEQ_LENGTH
else:
    sft_kwargs["max_length"] = MAX_SEQ_LENGTH

sft_config = SFTConfig(**sft_kwargs)

# If older TRL without completion_only_loss, fallback to DataCollatorForCompletionOnlyLM
collator = None
if not has_completion_loss:
    from trl import DataCollatorForCompletionOnlyLM
    collator = DataCollatorForCompletionOnlyLM(response_template="### Response:\n", tokenizer=tokenizer)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    data_collator=collator
)

t0 = time.time()
trainer.train(resume_from_checkpoint=last_checkpoint)
wall_time = time.time() - t0
peak_mem = torch.cuda.max_memory_allocated() / (1024**3) if torch.cuda.is_available() else 0.0
steps = trainer.state.global_step
steps_sec = steps / wall_time if wall_time > 0 else 0.0

print(f"\n--- Training Output Summary ---")
print(f"Wall time:       {wall_time:.2f}s")
print(f"Global steps:    {steps}")
print(f"Steps/sec:       {steps_sec:.3f}")
print(f"Peak GPU Memory: {peak_mem:.2f} GB")
print(f"Bodies Truncated:{body_truncated_count} ({body_truncated_count / len(df) * 100:.2f}%)")
if SMOKE_TEST:
    proj_sec = 3878 / steps_sec if steps_sec > 0 else 0
    print(f"Projected full run (3,878 steps): {proj_sec/3600:.2f} hours")

# First 5 Validation Predictions
print("\n--- First 5 Validation Predictions (Smoke Test Sanity) ---")
model.eval()
for i in range(5):
    v_row = val_df.iloc[i]
    prompt_text = build_prompt_pair(v_row['subject'], v_row['body'], int(v_row['label']), tokenizer=tokenizer)['prompt']
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    gen_text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    true_label = COMPLETION_PHISHING if v_row['label'] == 1 else COMPLETION_LEGITIMATE
    print(f"Val [{i+1}] True: {true_label} | Raw Output: {repr(gen_text)}")


In [ ]:
# CELL 10 — Save the LoRA adapter to Drive (small — a few MB, not the full model)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved adapter to:", OUTPUT_DIR)
print("This is on Google Drive, so it survives a Colab disconnect.")


## Quick sanity-check inference

Run this after training to eyeball a couple of predictions before moving to the full evaluation harness (Week 8).


In [ ]:
# CELL 11 — Quick inference smoke test
model.eval()
test_email = "Subject: Your package could not be delivered\n\nBody: Click here within 12 hours to reschedule delivery or your package will be returned to sender."
prompt = f"### Instruction:\nClassify the following email as 'Phishing' or 'Legitimate'.\n\n### Email:\n{test_email}\n\n### Response:\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=120, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))


---
### Reloading a saved adapter later (e.g. new Colab session, evaluation script)

```python
from peft import PeftModel
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto")
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)   # OUTPUT_DIR = your Drive path
```

### Next steps
- Replace the synthetic placeholder with your real `phishing_dataset.csv` from Week 2 — put it in Drive at `phishing-slm-research/data/processed/` so this notebook finds it automatically via `DRIVE_ROOT`.
- Run this notebook twice per model — `USE_COT = False` then `USE_COT = True` — to produce the naive and CoT-distilled arms for Table 3.
- For the CoT arm, first generate the `reasoning` column with a teacher model (Week 6's distillation step) — ask for that script when you're ready.
- Keep evaluation (accuracy/F1/AUROC) in a separate `eval/evaluate.py`, not this training notebook.
